# Goal

Получение нового бейзлайна на основе `VisionHead` версии 183 не получилась. Ну вернее, так, скорее всего требуется очень много запусков, т.к. видимо дело в стохастичности. Никакими приседаниями не удалось быстро родить выдающихся агентов, хотя попробовал много разных комбинаций. 

Возвращаясь к идее получения нового бейзлайна. Это всё было в контексте другой задачи - снабдить агента более длинным горизонтом наблюдений. Думалось, чтобы дать ему доступ к истории вплоть до 5 секунд. Это позволит ему не только высокочастотные зависимости познавать, но более долговременные. И таким образом агент по идее должен стать более разумным и креативным. Поэтому и хотелось получить чистый бейзлайн, в идеале из прогона какого-нибудь суперисследования. Чтобы потом можно было объективно сравнивать. Сравнивать агентов с более длинным горизонтом с бейзлайном.

Но, придётся, ехать без этого бейзлайна. Это исследование - просто проверить как будет работать `sequence_length=16`.

# TARGET_NOTEBOOK_FNAME

In [63]:
TARGET_NOTEBOOK_FNAME = '18n_ppo_tr_frostbite_08.ipynb'

# GRID_SEARCH_SPACE

In [64]:
# @launchit.collect
# GRID_SEARCH_SPACE = dict(
# )

# set_hyperparameters

In [65]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    import random
    HP.general.random_seed = random.randint(1, 100)
    HP.general.is_torch_deterministic = True
    HP.general.is_torch_compile = True
    HP.general.is_torch_amp = True
    HP.general.is_torch_amp_on_rollout = True
    
    HP.env.count = 32 
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6

    parent = '18d_world_model_09:183'
    HP.vision_head.parent = dict(model=parent, weights=parent)
    HP.vision_head.is_trainable = False
    
    HP.encoder.parent = dict(model=parent, weights=parent)
    HP.encoder.is_trainable = True

    HP.agent.parent = None 
    HP.agent.sequence_length = 16
    HP.agent.action_plan_length = 10
    HP.agent.d_model = 256 
    HP.agent.transformer = dict(layers_count=3, heads_count=4, attention_backend=['EFFICIENT_ATTENTION', 'MATH'])
    HP.agent.is_trainable = True
    
    # Test params
    HP.test.env_rams = [
        'com.develorium.neurolab.frostbite_ram:level1:1:cls=none',
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none',
    ]
    HP.test.env_ram_patches = [
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_right_of_igloo', 'bear_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_left_of_igloo', 'bear_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_right_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_20', 'bailey_near_center'], # *
    ]
    HP.test.break_on_level_passed = True
    
    # Training procedure params (PPO related) 
    HP.ppo.global_steps_count = 3_000_000 # total number of steps 
    HP.ppo.rollout_steps_count = 512 # how many steps to run in a single policy rolllout
    HP.ppo.rollout_env_rams = [
        'com.develorium.neurolab.frostbite_ram:level1:1:cls=none',
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none',
    ]
    HP.ppo.rollout_env_ram_patches = [
        ['no_score', 'last_life', 'full_igloo',            'temperature_10', 'bailey_right_at_the_igloo_door'], # 0 
        ['no_score', 'last_life', 'full_igloo',            'temperature_10', 'bailey_very_near_igloo_door'], # 1
        ['no_score', 'last_life', 'full_igloo',            'temperature_10', 'bailey_near_center'], # 2
        ['no_score', 'last_life', 'one_remaining_igloo',   'temperature_10', 'bailey_near_center'], # 3
        ['no_score', 'last_life', 'three_remaining_igloo', 'temperature_20', 'bailey_near_center', ], # 4
        ['no_score', 'last_life', 'half_igloo', 'bailey_near_center'], # 5
        ['no_score', 'last_life', 'half_igloo'], # 6

        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_right_of_igloo', 'bear_to_the_left_of_igloo'], # 7
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_left_of_igloo', 'bear_to_the_left_of_igloo'], # 8
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_right_of_igloo'], # 9
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_left_of_igloo'], # 10
        ['no_score', 'last_life', 'full_igloo', 'temperature_20', 'bailey_near_center'], # 11
    ]
    HP.ppo.rollout_env_stories = [
        '0;0:7',  # level 1 - teach agent to enter complete igloo
        '1;7:12', # level 5 - teach agent to enter complete blinking igloo
    ]
    
    HP.ppo.epochs_count = 2 
    HP.ppo.batch_size = 256 
    HP.ppo.learn_rate = 'const(0.00025)'
    HP.ppo.optimizer = 'AdamW'

    HP.ppo.is_obs_usage_enforced = False
    
    HP.ppo.ob_features_dropout = 0
    HP.ppo.ob_features_gauss_noise = 0
    
    HP.ppo.value_loss = dict(coef=0.2)
    HP.ppo.entropy_loss = dict(coef='const(0.05)')
    HP.ppo.consistency_loss = dict(type='MSE', coef=0.1)
    HP.ppo.prediction_loss = dict(coef=0.1)
    
    HP.ppo.gamma = 0.997 # return discount factor gamma
    HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
    HP.ppo.clip_coef = 0.1 # the surrogate clipping coefficient
    HP.ppo.clip_vloss = True
    HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
    HP.ppo.target_kl = None # the target KL divergence threshold
    HP.ppo.norm_adv = True # Toggles advantages normalization
    return HP

# Results

Видно, что `36.1` чуть хуже, чем `35.1` и `31.1`. При этом жрёт памяти где-то под 7 гигов и выполняется почти 1.5 часа вместо ~45 минут. Видно, что 16 обзов - это многовато. Надо рассматривать 8.

| Metric | Experiment `18n_study_36.1` (seq_len=16, `frostbite_08`) | Experiment `18n_study_35.1` (seq_len=4) | Experiment `18n_study_31.1` (seq_len=4) | Statistical Result / Verdict |
| :--- | :--- | :--- | :--- | :--- |
| **Sample Size (n)** | 18 | 18 | 24 | — |
| **Mean Score** | 0.5561 | 0.6583 | 0.6272 | −0.1023 vs `35.1`, −0.0711 vs `31.1` |
| **Best Value** | 0.759375 (`25`) | 0.896875 (`956`) | 0.796875 (`521`, `528`) | — |
| **Standard Deviation** | 0.0974 | 0.1446 | 0.0881 | — |
| **Shapiro-Wilk (p-value)** | 0.1575 (нормальное) | 0.8389 (нормальное) | 0.1066 (нормальное) | все распределения близки к нормальному |
| **Welch's t-test (p-value)** | — | — | — | **0.0187** vs `35.1` (значимо); **0.0200** vs `31.1` (значимо) |
| **Mann-Whitney U (p-value)** | — | — | — | **0.0367** vs `35.1` (значимо); **0.0167** vs `31.1` (значимо) |
| **Cohen's d (Effect Size)** | — | — | — | **−0.83** vs `35.1` (средне-крупный, в пользу `35.1`); **−0.77** vs `31.1` (средний, в пользу `31.1`) |

**Сравнение при общем родителе: `36.1` vs рука `:183` из `31.1`** (наиболее чистое сравнение — одинаковый родитель `18d_world_model_09:183`)

| Metric | `36.1` (n=18) | Рука `:183` из `31.1` (n=13) | Statistical Result / Verdict |
| :--- | :--- | :--- | :--- |
| **Mean Score** | 0.5561 | 0.6707 | −0.1146 в пользу `31.1` |
| **Best Value** | 0.759375 (`25`) | 0.796875 (`521`, `528`) | — |
| **Standard Deviation** | 0.0974 | 0.0867 | — |
| **Shapiro-Wilk (p-value)** | 0.1575 (нормальное) | 0.1812 (нормальное) | обе — нормальные |
| **Welch's t-test (p-value)** | — | — | **0.0018** (значимо, в пользу `31.1`) |
| **Mann-Whitney U (p-value)** | — | — | **0.0023** (значимо) |
| **Cohen's d (Effect Size)** | — | — | **−1.23** (большой, в пользу `31.1`) |

Вывод по статистике: `36.1` значимо хуже обоих бейзлайнов по всем тестам (все p < 0.05, эффект средний-крупный). На общем родителе (`:183`) разрыв максимальный — d≈1.23. Лучший триал `36.1` (`25`, 0.759375) ниже лучших обоих бейзлайнов (`956` — 0.896875, `521`/`528` — 0.796875); 16 из 18 триалов `36.1` ниже средних обоих бейзлайнов.

**Выводы**
1) `sequence_length=16` в `frostbite_08` значимо хуже бейзлайна с `sequence_length=4`: `36.1` (mean 0.5561, best `25` — 0.759375) проигрывает `35.1` (Welch p=0.0187, MWU p=0.0367, d≈−0.83) и `31.1` (Welch p=0.0200, MWU p=0.0167, d≈−0.77); на общем родителе (`31.1`, рука `:183`) эффект большой (d≈−1.23).
2) Оговорка: `36.1` отличалась от бейзлайнов не только `sequence_length` — ещё `batch_size=256` (против 512) и нотбук `frostbite_08` (рефакторинг). Чистое разнесение дельты по `sequence_length` требует контрольного прогона с `batch_size=512`, но направление и величина эффекта (16/18 триалов ниже средних бейзлайнов, best ниже best) маловероятно объясняются этими параметрами.
3) таргетим `sequence_length=8`

# System

In [66]:
import os, sys, re, subprocess, json
import IPython 
import concurrent.futures as cf
from collections import namedtuple

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
from optuna.trial import TrialState

project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]

sys.path.append(os.path.join(project_root_path, 'lib'))

from logging_utils import *
from math_utils import *
from artifact_registry import *
import launchit
import launch_dispatcher
from autoincrement import Autoincrement

In [67]:
CONFIG = namedtuple('CONFIG', 
                    'project_root_uri, model_group_uri, project_root_path, subproject_name, subproject_path, run_path, ' + 
                    'target_notebook_fname, target_notebook_name, ' + 
                    'optuna_study_notebook_fname, optuna_study_name, optuna_study_serial, optuna_study_fname')(
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    model_group_uri=None,
    project_root_path=project_root_path,
    subproject_name=None,
    subproject_path=os.path.abspath('../..'),
    run_path=None,
    target_notebook_fname=os.path.join(os.path.abspath('../..'), TARGET_NOTEBOOK_FNAME),
    target_notebook_name=None,
    optuna_study_notebook_fname=None,
    optuna_study_name=None,
    optuna_study_serial=None,
    optuna_study_fname=None,
)

with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as connection_file:
    optuna_study_notebook_fname = json.load(connection_file).get('jupyter_session')
    optuna_study_name, _ = os.path.splitext(os.path.basename(optuna_study_notebook_fname))
    optuna_study_serial = re.match(r'\w+_([\d\.\w]+)', optuna_study_name).group(1)
    optuna_study_fname = os.path.join(os.path.dirname(optuna_study_notebook_fname), optuna_study_name + '.optuna')
    CONFIG = CONFIG._replace(optuna_study_notebook_fname=optuna_study_notebook_fname)
    CONFIG = CONFIG._replace(optuna_study_name=optuna_study_name)
    CONFIG = CONFIG._replace(optuna_study_serial=optuna_study_serial)
    CONFIG = CONFIG._replace(optuna_study_fname=optuna_study_fname)

target_notebook_name, _ = os.path.splitext(os.path.basename(TARGET_NOTEBOOK_FNAME))
CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.target_notebook_fname)))
CONFIG = CONFIG._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
CONFIG = CONFIG._replace(target_notebook_name=target_notebook_name)
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG._asdict()

{'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'subproject_name': '18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'target_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/18n_ppo_tr_frostbite_08.ipynb',
 'target_notebook_name': '18n_ppo_tr_frostbite_08',
 'optuna_study_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_36.1/18n_study_36.1.ipynb',
 'optuna_study_name': '18n_study_36.1',
 'optuna_study_serial': '36.1',
 'optuna_study_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_36.1/18n_study_36.1.optuna'}

In [68]:
LOG = Logging.get()
LOG.enable('syslog', False)
LOG.enable('stdout', False)
LOG.enable('verbose_stdout', True)

In [69]:
ARTIFACT_REGISTRY = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri)

In [70]:
def create_optuna_launch():
    model_version = int(Autoincrement.get(f'{CONFIG.model_group_uri}.{CONFIG.target_notebook_name}'))
    assert model_version > 0, model_version
    ARTIFACT_REGISTRY.register_component(CONFIG.target_notebook_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    # Prep docker launch
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.target_notebook_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL='TRAIN',
        OPTUNA_STUDY_FNAME=CONFIG.optuna_study_fname,
        OPTUNA_STUDY_NAME=CONFIG.optuna_study_name,
        OPTUNA_DECISIVE_METRIC='test/levels_passed_mean',
    )
    launch_fname = launchit.launchit(
        CONFIG.target_notebook_fname, 
        launch_serial=int(model_version),
        expandvars=expandvars, 
        make_py_file=False, 
        dir_name=CONFIG.run_path,
        collect_inds=['temp_config', 'optuna', 'initrd', 'build_docker_launch', 'optuna_run_docker_launch'],
        disable_inds=[],
    )
    return f'{CONFIG.target_notebook_name}:{model_version}', launch_fname

In [71]:
# Executed in a separate thread with GIL locked
def run_optuna_launch(launch_fname):
    # Run launch notebook locally, the latter will:
    # 1) sample values of hyperparameters from optuna study
    # 2) pack everything to docker launch (self-contained thing)
    # 3) run "docker_launch_run" cell which in turn will dispatch launch to cloud via launch_dispatcher
    # 4) collect result of a docker launch from cloud and update optuna study
    LOG(f'Launching "{launch_fname}"')
    
    subprocess.run(
        ['papermill', launch_fname, launch_fname, '--no-progress-bar'],
        capture_output=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    if os.path.exists(launch_fname + '.out'):
        with open(launch_fname + '.out', 'rt') as f:
            LOG(f.read())
    else:
        LOG(f'"{launch_fname}" completed with no output, probably failed')

## Unleash!

In [72]:
optuna_study = optuna.create_study(
    study_name=CONFIG.optuna_study_name,
    directions=['maximize'],
    storage=JournalStorage(JournalFileBackend(file_path=CONFIG.optuna_study_fname)),
    load_if_exists=True,
)
optuna_study.set_user_attr('STUDY_SERIAL', CONFIG.optuna_study_serial)
launches_count = 18
completed_launches_count = 0

with LOG.auto_log_level(logging.INFO):
    with cf.ThreadPoolExecutor(max_workers=32) as executor:
        futures = {}
        idle_runners_af = RecursiveMovingAverageFilter(max_n=6)
        is_first_time = True
        
        while launches_count is None or completed_launches_count < launches_count:
            runners_info = launch_dispatcher.RunnersInfo.get()
            idle_runners_af(runners_info['idle'])

            if is_first_time or (idle_runners_af.n >= idle_runners_af.max_n and idle_runners_af.v >= 1):
                if launches_count is None or (completed_launches_count + len(futures) < launches_count):
                    launch_name, launch_fname = create_optuna_launch()
                    futures.update({executor.submit(run_optuna_launch, launch_fname): launch_name})
                    LOG(f'{idle_runners_af.v:.1f} idle runners exist, submitted launch "{launch_name}"; running launches={len(futures)}')
                    idle_runners_af.reset()
                    
                is_first_time = False

            try:
                while futures:
                    completed_futures, _ = cf.wait(futures, timeout=0.1, return_when=cf.FIRST_COMPLETED)

                    if not completed_futures:
                        break
                        
                    for completed_future in completed_futures:
                        launch_name = futures[completed_future]
                        del futures[completed_future]

                        exc = completed_future.exception()
                        
                        if exc is not None:
                            LOG(f'Launch "{launch_name}" failed: {exc}')
                        else:
                            LOG(f'Launch "{launch_name}" completed')
    
                    if completed_futures:
                        completed_launches_count += len(completed_futures)
                        LOG(f'{completed_launches_count} (+{len(completed_futures)}) launches completed; running launches={len(futures)}')
            except TimeoutError as e:
                pass

            time.sleep(5)

[I 2026-10-09 23:05:28,684] A new study created in Journal with name: 18n_study_36.1


2026.10.09-23:05:29.006382     0.290 >> Model instance registered, version=12
2026.10.09-23:05:29.019823     0.004 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_08-launch12.ipynb"
2026.10.09-23:05:29.020059     0.004 >> 6.0 idle runners exist, submitted launch "18n_ppo_tr_frostbite_08:12"; running launches=1
2026.10.09-23:06:00.035817     0.324 >> Model instance registered, version=13
2026.10.09-23:06:00.045718     0.004 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_08-launch13.ipynb"
2026.10.09-23:06:00.047450     0.002 >> 5.3 idle runners exist, submitted launch "18n_ppo_tr_frostbite_08:13"; running launches=2
2026.10.09-23:06:31.109710     0.353 >> Model instance registered, version=14
2026.10.09-23:06:31.118937     0.004 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_08-launch14.ipynb"
2026.10.09-23:06:31.119300     0.005 >> 2.8 idle runners exist, submitted launch "18n_ppo_tr_frostbite_08:14"; ru

In [73]:
study = optuna.create_study(
    study_name=optuna_study_name,
    storage=JournalStorage(JournalFileBackend(file_path=optuna_study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs.get('MODEL_VERSION', 'n/a')}')
    
    LOG('\tParams: ')
    
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    LOG(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        LOG(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        LOG(f"\tnumber: {trial.number}")
        LOG(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        LOG(f"\tparams: {trial.params}")
        LOG(f"\tvalues: {trial.values}")

[I 2026-10-10 06:39:41,140] Using an existing study with name '18n_study_36.1' instead of creating a new one.


2026.10.10-06:39:41.143483     5.040 >> Study statistics: 
2026.10.10-06:39:41.147642     0.004 >> 	Number of finished trials: 18
2026.10.10-06:39:41.148820     0.001 >> 	Number of pruned trials: 0
2026.10.10-06:39:41.149976     0.001 >> 	Number of complete trials: 18
2026.10.10-06:39:41.151014     0.001 >> Best trial:
2026.10.10-06:39:41.151983     0.001 >> 	Value: 0.759375
2026.10.10-06:39:41.152325     0.000 >> 	Model version: 25
2026.10.10-06:39:41.152610     0.000 >> 	Params: 


In [74]:
top_trials = sorted(filter(lambda t: t.values, study.trials), key=lambda t: -t.values[0])[:3]
LOG(f'Top trials:')

for i, trial in enumerate(top_trials):
    LOG(f'{i+1}) {CONFIG.target_notebook_name + ':' + trial.user_attrs['MODEL_VERSION']}, value={trial.values[0]}')

2026.10.10-06:39:41.159460     0.007 >> Top trials:
2026.10.10-06:39:41.160068     0.001 >> 1) 18n_ppo_tr_frostbite_08:25, value=0.759375
2026.10.10-06:39:41.160332     0.000 >> 2) 18n_ppo_tr_frostbite_08:23, value=0.728125
2026.10.10-06:39:41.160544     0.000 >> 3) 18n_ppo_tr_frostbite_08:17, value=0.6


In [75]:
values = []

for trial in study.trials:
    if trial.values is not None:
        values.append(trial.values[0])

values = np.array(values)
values.mean(), values.std()

(np.float64(0.5560763888888889), np.float64(0.0946683734706105))

In [76]:
','.join(map(str, values.tolist()))

'0.521875,0.596875,0.4,0.596875,0.5,0.6,0.6,0.5125,0.515625,0.596875,0.496875,0.728125,0.590625,0.759375,0.59375,0.578125,0.428125,0.39375'